<a href="https://colab.research.google.com/github/nuhuynhh/AAI2026/blob/main/PromptEngExercise1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Exercise 1 - Prompt Chain for a Customer Support AI
Tool Used: Open API in Google Colab

In [ ]:
!pip -q install requests

In [ ]:
import os
from getpass import getpass
import requests

os.environ["LANGBASE_API_KEY"] = getpass("Enter your LANGBASE API key: ")
LANGBASE_API_KEY = os.environ["LANGBASE_API_KEY"]


Enter your LANGBASE API key: ··········


In [ ]:
def run_pipe(pipe_name: str, variables: dict, user_message: str = "Run this pipe."):
    url = "https://api.langbase.com/v1/pipes/run"
    headers = {
        "Authorization": f"Bearer {LANGBASE_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "name": pipe_name,
        "variables": variables,
        "messages": [{"role": "user", "content": user_message}],
    }

    resp = requests.post(url, headers=headers, json=payload)

    # Always show status + text if not JSON
    print("HTTP Status:", resp.status_code)

    # If response isn't JSON, this will help us see what's going on
    content_type = resp.headers.get("Content-Type", "")
    print("Content-Type:", content_type)

    # Print first 500 chars to avoid huge spam
    print("Raw text (first 500 chars):", resp.text[:500])

    # Only attempt JSON parsing if it looks like JSON
    if "application/json" in content_type.lower():
        return resp.json()
    else:
        # Return a structured object so your notebook doesn't crash
        return {"success": False, "status_code": resp.status_code, "raw_text": resp.text}

In [ ]:
# Step 1 — Classification

customer_message = "My dashboard won’t load and I keep getting a 504 error."

r1 = run_pipe(
    pipe_name="support-classify",
    variables={"customer_message": customer_message},
    user_message=customer_message
)

print("Parsed Response Object:", r1)

HTTP Status: 200
Content-Type: text/event-stream
Raw text (first 500 chars): data: {"id":"1772004696226","object":"chat.completion.chunk","created":1772004696,"model":"gemini-2.5-flash","provider":"google","choices":[{"delta":{"role":"assistant","content":"Technical"},"index":0,"finish_reason":"STOP"}],"usage":{"prompt_tokens":91,"completion_tokens":1,"total_tokens":129}}


Parsed Response Object: {'success': False, 'status_code': 200, 'raw_text': 'data: {"id":"1772004696226","object":"chat.completion.chunk","created":1772004696,"model":"gemini-2.5-flash","provider":"google","choices":[{"delta":{"role":"assistant","content":"Technical"},"index":0,"finish_reason":"STOP"}],"usage":{"prompt_tokens":91,"completion_tokens":1,"total_tokens":129}}\n\n'}


In [ ]:
category = (r1.get("completion") or r1.get("output") or "").strip()
print("Category:", category)

Category: 


In [ ]:
def extract_text(resp: dict) -> str:
    # Try common fields
    for key in ["completion", "output", "text", "message"]:
        if isinstance(resp, dict) and resp.get(key):
            return str(resp.get(key)).strip()
    # Fallback: show whole dict
    return ""

category = extract_text(r1)

if not category:
    # Temporary fallback so you can keep progressing (replace once Step 1 works)
    category = "Technical"

print("Step 1 Category:", category)

Step 1 Category: Technical


In [ ]:
# Step 2 — Clarifying Questions (uses Step 1 category)

r2 = run_pipe(
    pipe_name="support-questions",
    variables={
        "customer_message": customer_message,
        "category": category
    },
    user_message=f"Category: {category}\nCustomer message: {customer_message}"
)

print("Step 2 Raw Response:", r2)

questions = extract_text(r2)
print("\nStep 2 Output (Questions):\n", questions)

HTTP Status: 400
Content-Type: application/json
Raw text (first 500 chars): {"success":false,"error":{"code":"BAD_REQUEST","status":400,"message":"Missing LLM API key for 'OpenAI'. Add on Langbase.com","docs":"https://langbase.com/docs/api-reference/errors/bad_request"}}
Step 2 Raw Response: {'success': False, 'error': {'code': 'BAD_REQUEST', 'status': 400, 'message': "Missing LLM API key for 'OpenAI'. Add on Langbase.com", 'docs': 'https://langbase.com/docs/api-reference/errors/bad_request'}}

Step 2 Output (Questions):
 


In [ ]:
# Step 3 — Simulated Customer Answers

customer_answers = "I am using Chrome. The issue started today after logging in."

r3 = run_pipe(
    pipe_name="support-solution",
    variables={
        "customer_message": customer_message,
        "category": category,
        "customer_answers": customer_answers
    },
    user_message=f"""
Category: {category}
Customer message: {customer_message}
Customer answers: {customer_answers}
"""
)

print("Step 3 Raw Response:", r3)

solution = extract_text(r3)
print("\nStep 3 Output (Solution):\n", solution)

HTTP Status: 400
Content-Type: application/json
Raw text (first 500 chars): {"success":false,"error":{"code":"BAD_REQUEST","status":400,"message":"Missing LLM API key for 'OpenAI'. Add on Langbase.com","docs":"https://langbase.com/docs/api-reference/errors/bad_request"}}
Step 3 Raw Response: {'success': False, 'error': {'code': 'BAD_REQUEST', 'status': 400, 'message': "Missing LLM API key for 'OpenAI'. Add on Langbase.com", 'docs': 'https://langbase.com/docs/api-reference/errors/bad_request'}}

Step 3 Output (Solution):
 


In [ ]:
# Step 4 — Escalation Decision

r4 = run_pipe(
    pipe_name="support-escalation",
    variables={
        "category": category,
        "solution_text": solution
    },
    user_message=f"""
Category: {category}
Proposed solution: {solution}
"""
)

print("Step 4 Raw Response:", r4)

decision = extract_text(r4)
print("\nStep 4 Output (Decision):", decision)

HTTP Status: 400
Content-Type: application/json
Raw text (first 500 chars): {"success":false,"error":{"code":"BAD_REQUEST","status":400,"message":"Invalid request error. Often missing required parameters or typo. HINT: Validation error: Variable value cannot be empty at \"variables.solution_text\"","docs":"https://langbase.com/docs/api-reference/errors/bad_request"}}
Step 4 Raw Response: {'success': False, 'error': {'code': 'BAD_REQUEST', 'status': 400, 'message': 'Invalid request error. Often missing required parameters or typo. HINT: Validation error: Variable value cannot be empty at "variables.solution_text"', 'docs': 'https://langbase.com/docs/api-reference/errors/bad_request'}}

Step 4 Output (Decision): 
